# M0 — Word TF-IDF + Naïve Bayes 

Ce modèle utilise une représentation TF-IDF des énoncés, puis un classifieur Naïve Bayes multinomial pour prédire les huit tags de chaque exercice. Plusieurs configurations d'hyperparamètres sont ensuite comparées sur l'ensemble de validation afin de sélectionner la meilleure selon le F1 macro. Les seuils de décision sont ajustés pour chaque tag à partir des quantiles des probabilités prédites, puis le modèle est évalué et sauvegardé. L'ensemble de test reste réservé à l'évaluation finale.

## 1. Charger et contrôler les données

Les descriptions, entrées, sorties et notes forment les seules variables explicatives. Le code, les identifiants et les tags ne sont pas inclus dans le texte.

In [9]:
import hashlib
import html
import json
import platform
import unicodedata
import warnings
from importlib.metadata import version
from pathlib import Path
from time import perf_counter

import joblib
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.exceptions import ConvergenceWarning
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import (
    accuracy_score, f1_score, hamming_loss,
    precision_recall_fscore_support,
)
from sklearn.multiclass import OneVsRestClassifier
from threadpoolctl import threadpool_limits

racine = next(
    (p for p in [Path.cwd(), *Path.cwd().parents]
     if (p / "data" / "entrainement.jsonl").is_file()), None
)
if racine is None:
    raise FileNotFoundError("Ouvrez ce notebook depuis le dépôt du projet.")

In [10]:
dossier_modele = racine / "models" / "m0_word_tfidf_naive_bayes"
dossier_modele.mkdir(parents=True, exist_ok=True)
graine = 42
tags_cibles = ["math", "graphs", "strings", "number theory",
               "trees", "geometry", "games", "probabilities"]
colonnes_cibles = ["cible_" + tag.replace(" ", "_") for tag in tags_cibles]
champs_texte = {"prob_desc_description": "DESCRIPTION", "prob_desc_input_spec": "ENTREES",
                "prob_desc_output_spec": "SORTIES", "prob_desc_notes": "NOTES"}
versions = {nom: version(nom) for nom in
            ["numpy", "pandas", "scipy", "scikit-learn", "matplotlib", "joblib", "threadpoolctl"]}
empreintes = {}

def charger_ensemble(nom):
    chemin = racine / "data" / f"{nom}.jsonl"
    contenu = chemin.read_bytes()
    empreintes[nom] = hashlib.sha256(contenu).hexdigest()
    return pd.DataFrame([json.loads(ligne) for ligne in contenu.decode("utf-8").splitlines()
                         if ligne.strip()])

entrainement = charger_ensemble("entrainement")
validation = charger_ensemble("validation")
for donnees in [entrainement, validation]:
    assert donnees[colonnes_cibles].isin([0, 1]).all().all(), "Cibles non binaires."
    assert set(champs_texte).issubset(donnees.columns)
y_train = entrainement[colonnes_cibles].to_numpy(dtype=np.int64)
y_validation = validation[colonnes_cibles].to_numpy(dtype=np.int64)
for champ in ["fichier_source", "src_uid", "code_uid"]:
    assert not entrainement[champ].isna().any() and not validation[champ].isna().any()
    assert not entrainement[champ].duplicated().any()
    assert not validation[champ].duplicated().any()
    assert set(entrainement[champ]).isdisjoint(validation[champ]), f"Fuite : {champ}"
for cibles in [y_train, y_validation]:
    assert ((cibles.sum(axis=0) > 0) & (cibles.sum(axis=0) < len(cibles))).all()

warnings.filterwarnings("error", category=ConvergenceWarning)
print(f"{len(entrainement)} exercices d’entraînement et {len(validation)} de validation.")

3470 exercices d’entraînement et 739 de validation.


## 2. Construire le Word TF-IDF

Pour chaque exercice, la description, les spécifications d’entrée et de sortie et les notes sont réunies en un seul texte. Les champs vides sont ignorés et les caractères HTML et Unicode sont harmonisés. Le TF-IDF transforme ce texte en valeurs numériques à partir des mots et des paires de mots consécutifs. Le vocabulaire et les pondérations sont calculés uniquement sur l’ensemble d’entraînement, puis appliqués à la validation.

In [11]:
def construire_enonce(exercice):
    sections = []
    for champ, titre in champs_texte.items():
        valeur = exercice.get(champ)
        if isinstance(valeur, str) and valeur.strip():
            contenu = unicodedata.normalize("NFC", html.unescape(valeur)).strip()
            sections.append(f"{titre}\n{contenu}")
    return "\n\n".join(sections)

textes_train = entrainement.apply(construire_enonce, axis=1)
textes_validation = validation.apply(construire_enonce, axis=1)
assert textes_train.str.strip().ne("").all() and textes_validation.str.strip().ne("").all()

parametres_tfidf = {
    "ngram_range": (1, 2), "min_df": 2, "max_df": 0.98, "max_features": 80000,
    "sublinear_tf": True, "norm": "l2", "lowercase": True,
    "token_pattern": r"(?u)\b\w+\b",
}
vectoriseur = TfidfVectorizer(**parametres_tfidf)
X_train = vectoriseur.fit_transform(textes_train)
X_validation = vectoriseur.transform(textes_validation)
assert (X_train.getnnz(axis=1) > 0).all()
print(f"{X_train.shape[1]:,} variables TF-IDF.")
display(pd.Series(parametres_tfidf, name="Valeur").to_frame())

71,774 variables TF-IDF.


,Valeur
ngram_range,"(1, 2)"
min_df,2
max_df,0.98
max_features,80000
sublinear_tf,True
norm,l2
lowercase,True
token_pattern,(?u)\b\w+\b


## 3. Comparer six configurations

## 3. Comparer six configurations

Huit classifieurs Naïve Bayes multinomiaux sont entraînés via `OneVsRestClassifier`.
Les six couples `alpha ∈ {0.01, 0.1, 1}` et `fit_prior ∈ {True, False}`
sont comparés avec le seuil par défaut de 0,5. `alpha` contrôle le lissage
et `fit_prior` la prise en compte des fréquences des classes. Le F1 macro donne le même poids aux huit tags, le F1 micro complète la lecture globale.

In [12]:
def evaluer_predictions(reels, predictions):
    return {
        "f1_macro": float(f1_score(reels, predictions, average="macro", zero_division=0)),
        "f1_micro": float(f1_score(reels, predictions, average="micro", zero_division=0)),
        "hamming_loss": float(hamming_loss(reels, predictions)),
        "exact_match_ratio": float(accuracy_score(reels, predictions)),
    }

In [13]:
seuil_defaut = 0.5

def creer_classifieur(alpha, fit_prior):
    estimateur = MultinomialNB(alpha=alpha, fit_prior=fit_prior)
    return OneVsRestClassifier(estimateur, n_jobs=1)

def calculer_scores(classifieur, matrice):
    return classifieur.predict_proba(matrice)

experiences = []
classifieurs = {}
with threadpool_limits(limits=1):
    for alpha in [0.01, 0.1, 1.0]:
        for nom_prior, fit_prior in [("appris", True), ("uniforme", False)]:
            cle = f"alpha{alpha}_{nom_prior}"
            classifieur = creer_classifieur(alpha, fit_prior)
            debut = perf_counter()
            classifieur.fit(X_train, y_train)
            duree = perf_counter() - debut
            scores = calculer_scores(classifieur, X_validation)
            predictions = (scores >= seuil_defaut).astype(int)
            classifieurs[cle] = classifieur
            experiences.append({"experience": cle, "alpha": alpha, "fit_prior": fit_prior,
                                 "temps_entrainement_s": duree,
                                 **evaluer_predictions(y_validation, predictions)})

recherche = pd.DataFrame(experiences).sort_values(
    ["f1_macro", "alpha"], ascending=[False, True], kind="stable"
).reset_index(drop=True)
configuration_retenue = recherche.iloc[0]
classifieur_final = classifieurs[configuration_retenue["experience"]]
scores_validation = calculer_scores(classifieur_final, X_validation)
display(recherche.round(4))
print("Configuration retenue :", configuration_retenue["experience"])


,experience,alpha,fit_prior,temps_entrainement_s,f1_macro,f1_micro,hamming_loss,exact_match_ratio
0,alpha0.01_uniforme,0.01,False,0.0260,0.5843,0.5571,0.0775,0.4926
1,alpha0.01_appris,0.01,True,0.0334,0.4835,0.5189,0.0709,0.5304
2,alpha0.1_uniforme,0.10,False,0.0248,0.3486,0.4811,0.0744,0.5088
3,alpha0.1_appris,0.10,True,0.0262,0.2165,0.3591,0.0743,0.5156
4,alpha1.0_appris,1.00,True,0.0271,0.0000,0.0000,0.0864,0.4587
5,alpha1.0_uniforme,1.00,False,0.0309,0.0000,0.0000,0.0864,0.4587


Configuration retenue : alpha0.01_uniforme


## 4. Ajuster les huit seuils

Le classifieur retenu est conservé sans réentraînement. Pour chaque tag, choisir
le seuil donnant le meilleur F1 parmi une grille de quantiles des scores de validation
et le seuil par défaut (0.5).

In [14]:
quantiles_seuils = [0.50, 0.70, 0.80, 0.85, 0.90, 0.925,
                   0.95, 0.965, 0.975, 0.985, 0.99, 0.995]
seuils = []
for indice in range(len(tags_cibles)):
    candidats = np.unique(np.append(
        np.quantile(scores_validation[:, indice], quantiles_seuils), seuil_defaut
    ))
    evaluations = [
        (f1_score(y_validation[:, indice], scores_validation[:, indice] >= seuil,
                  zero_division=0), float(seuil)) for seuil in candidats
    ]
    meilleur = min(evaluations, key=lambda r: (-r[0], abs(r[1] - seuil_defaut), r[1]))
    seuils.append(meilleur[1])
seuils = np.array(seuils)

## 5. Lire les résultats de validation

Comparer les seuils par défaut et ajustés, puis examiner chaque tag avec son support,
sa précision, son rappel et son F1. 

In [15]:
lignes_globales, lignes_tags = [], []
for nom, valeur in [("Aucun tag", 0), ("Tous les tags", 1)]:
    predictions = np.full_like(y_validation, valeur)
    lignes_globales.append({"regle": nom, **evaluer_predictions(
        y_validation, predictions
    )})
for regle, seuils_regle in [("Défaut", np.full(8, seuil_defaut)), ("Ajustés", seuils)]:
    predictions = (scores_validation >= seuils_regle).astype(int)
    lignes_globales.append({"regle": regle,
                           **evaluer_predictions(y_validation, predictions)})
    precisions, rappels, f1, supports = precision_recall_fscore_support(
        y_validation, predictions, average=None, zero_division=0
    )
    for i, tag in enumerate(tags_cibles):
        lignes_tags.append({
            "regle": regle, "tag": tag, "support": int(supports[i]),
            "seuil": float(seuils_regle[i]), "precision": precisions[i],
            "rappel": rappels[i], "f1": f1[i],
        })
bilan = pd.DataFrame(lignes_globales)
par_tag = pd.DataFrame(lignes_tags)
display(bilan.round(4))
display(par_tag.loc[par_tag["regle"].eq("Ajustés")].round(4).reset_index(drop=True))

sans_cible = y_validation.sum(axis=1) == 0
controle_sans_cible = []
for regle, seuils_regle in [("Défaut", np.full(8, seuil_defaut)), ("Ajustés", seuils)]:
    predictions = scores_validation[sans_cible] >= seuils_regle
    avec_prediction = predictions.any(axis=1)
    controle_sans_cible.append({
        "regle": regle,
        "exercices_sans_cible": int(sans_cible.sum()),
        "aucun_tag_predit": int((~avec_prediction).sum()),
        "au_moins_un_tag_predit": int(avec_prediction.sum()),
    })
print("Contrôle des exercices sans tag cible (décomptes) :")
display(pd.DataFrame(controle_sans_cible))

,regle,f1_macro,f1_micro,hamming_loss,exact_match_ratio
0,Aucun tag,0.0000,0.0000,0.0864,0.4587
1,Tous les tags,0.1500,0.1591,0.9136,0.0000
2,Défaut,0.5843,0.5571,0.0775,0.4926
3,Ajustés,0.6161,0.5763,0.0846,0.4425


,regle,tag,support,seuil,precision,rappel,f1
0,Ajustés,math,211,0.0790,0.4243,0.7441,0.5404
1,Ajustés,graphs,81,0.5805,0.6622,0.6049,0.6323
2,Ajustés,strings,63,0.5000,0.5811,0.6825,0.6277
3,Ajustés,number theory,52,0.4776,0.4821,0.5192,0.5000
4,Ajustés,trees,49,0.9951,0.7297,0.5510,0.6279
5,Ajustés,geometry,25,0.5102,0.6154,0.6400,0.6275
6,Ajustés,games,16,0.5000,0.9231,0.7500,0.8276
7,Ajustés,probabilities,14,0.0761,0.4737,0.6429,0.5455


Contrôle des exercices sans tag cible (décomptes) :


,regle,exercices_sans_cible,aucun_tag_predit,au_moins_un_tag_predit
0,Défaut,339,224,115
1,Ajustés,339,149,190


## 6. Sauvegarder le modèle final et les résultats

Un unique artefact contient le vectoriseur, les huit classifieurs, les seuils,
l’ordre des tags et les règles de construction du texte. Les trois CSV conservent
la recherche, les métriques globales et le détail par tag. Le modèle sauvegardé
reste celui appris sur l’entraînement, les seuils ont été réglés sur la validation.

In [16]:
modele_final = {
    "modele": "M0", "vectoriseur": vectoriseur, "classifieur": classifieur_final,
    "seuils": seuils, "tags": tags_cibles, "champs_texte": champs_texte,
    "type_scores": "probabilites",
    "pretraitement": {"html_unescape": True, "normalisation_unicode": "NFC",
                      "separateur_sections": "\n\n", "separateur_titre": "\n"},
    "configuration": {"alpha": float(configuration_retenue["alpha"]),
                      "fit_prior": bool(configuration_retenue["fit_prior"])},
    "graine": graine, "versions": versions, "python": platform.python_version(),
    "empreintes_donnees": empreintes,
}
_ = joblib.dump(modele_final, dossier_modele / "modele.joblib", compress=3)
recherche.to_csv(dossier_modele / "recherche_hyperparametres.csv", index=False)
bilan.to_csv(dossier_modele / "metriques_validation.csv", index=False)
par_tag.to_csv(dossier_modele / "metriques_par_tag_validation.csv", index=False)